# HIGHFLY · CHARACTER FACTORY · PRUEBA GRATUITA EN COLAB (R0.1)

**No instalar nada en tu PC.** Abrí este cuaderno en Google Colab, elegí una **GPU T4 gratuita** en «Entorno de ejecución → Cambiar tipo de entorno de ejecución» y tocá **«Ejecutar todo»**. Si Colab proporciona GPU y espacio suficientes, todo lo demás se hace automáticamente: instalar, verificar, generar una muestra y descargar un ZIP. No se usan APIs comerciales ni créditos.

**¡Importante!** Esto **NO** importa el Warrior ni crea su armadura definitiva: Character Factory v0.1 genera un **personaje nuevo con 127 huesos y una capa de ropa**, distinto al Warrior ClaudeCraft (23 huesos / 22 clips). Esta prueba solo determina si podemos ejecutar gratuitamente el generador y obtener sus mallas de prenda por separado para investigación. No sustituir el juego ni presentar como skin aprobada.

**Coste real:** Software abierto Apache-2.0, ejecución propia sin créditos. Requiere GPU CUDA compatible y **descarga inicial oficial de unos 36,4 GB de pesos**; Colab Free puede desconectar, quedarse sin memoria o limitar disponibilidad. El cuaderno comprueba GPU/espacio *antes de descargar*. El tamaño mínimo de disco reservado es conservador y puede excluir algunas sesiones que podrían funcionar.

**Salida si funciona:** `HIGHFLY_CHARACTERFACTORY_TRIAL_R01.zip` con `scene.glb`, `character.char.json` y `highfly-probe-report.json`. Ningún archivo es un wearable ClaudeCraft. Si falla, se muestra el error real sin inventar éxito.

**Código fijado:** `character-factory/character-factory` commit `379b59f1b40172aa318cf0f67d3529ded255f8ac`, instalación local en Python 3.11 aislado (evita los errores de Colab Python 3.13). El código se descarga de GitHub y los modelos públicos de Hugging Face; la inferencia queda en la GPU de Colab.

> Nota: GitHub Actions solo valida la sintaxis y las barreras de seguridad del cuaderno. **No prueba** que los 36 GB de pesos o la inferencia funcionen realmente en T4.

In [ ]:
# HIGHFLY · CHARACTER FACTORY R0.1 · UNA SOLA CELDA / SIN PAGOS
import os, sys, subprocess, shutil, json, zipfile, hashlib, traceback, collections, time
from pathlib import Path
from datetime import datetime, timezone

CHECKPOINT = "379b59f1b40172aa318cf0f67d3529ded255f8ac"
REPO = Path("/content/highfly-character-factory")
ENV = Path("/content/highfly-cf-python311")
PY = ENV / "bin/python"
BIN = ENV / "bin/character-factory"
OUTPUT = Path("/content/highfly-cf-obsidian-trial")
ZIP = Path("/content/HIGHFLY_CHARACTERFACTORY_TRIAL_R01.zip")
PROMPT = ("A fantasy dark knight with obsidian-black medieval armor and silver trims, "
          "violet glowing gemstones, dramatic angular helmet and ornate armored pauldrons, "
          "dark metal breastplate, gauntlets, greaves and heavy armored boots. "
          "Single humanoid character standing in a neutral stance; distinguish clothing "
          "and footwear as separate materials. No wings.")
stage_name = "inicio"
t0=time.time()

def stage(message):
    global stage_name
    stage_name = message
    print("\n" + "="*65 + "\nHIGHFLY ► " + stage_name + "\n" + "="*65, flush=True)

def run(args, cwd=None, timeout=900, env=None):
    args=[str(x) for x in args]
    print(">", " ".join(args), flush=True)
    recent=collections.deque(maxlen=80)
    with subprocess.Popen(args,cwd=cwd,env=env,stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT,text=True,bufsize=1,
                          errors="replace") as proc:
        try:
            for line in proc.stdout:
                print(line,end="",flush=True)
                recent.append(line.rstrip())
            rc=proc.wait(timeout=timeout)
        except BaseException:
            proc.kill(); proc.wait(); raise
    if rc:
        print("FALLÓ UN COMANDO. Últimas líneas:\n"+"\n".join(recent), flush=True)
        raise subprocess.CalledProcessError(rc,args)
    return True

try:
    stage("1/5 · GPU Y ESPACIO GRATIS (ANTES DE DESCARGAS GRANDES)")
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("Colab no tiene GPU CUDA. Elegí GPU T4 gratuita. No comprar recursos.")
    gpu=torch.cuda.get_device_properties(0)
    vram=gpu.total_memory/2**30
    disk_free=shutil.disk_usage("/content").free/2**30
    print("GPU:",gpu.name,"| VRAM GiB:",round(vram,2),
          "| Disco libre GiB:",round(disk_free,2), flush=True)
    if vram<12:
        raise RuntimeError("CHARACTER_FACTORY_GPU_TOO_SMALL: requiere como referencia >=12 GiB VRAM.")
    if disk_free<60:
        raise RuntimeError("CHARACTER_FACTORY_DISK_TOO_SMALL: exige >=60 GiB libres antes de bajar ~36.4 GB de pesos.")
    # La ruta utiliza intérprete LOCAL, sin endpoints/API comerciales.
    cleanenv=dict(os.environ)
    for key in ("OPENAI_API_KEY","ANTHROPIC_API_KEY","GEMINI_API_KEY",
                "CHARACTER_FACTORY_INTERPRETER_ENDPOINT",
                "CHARACTER_FACTORY_INTERPRETER_API_KEY",
                "CHARACTER_FACTORY_INTERPRETER_MODEL"):
        cleanenv.pop(key,None)
    cleanenv["PYTORCH_CUDA_ALLOC_CONF"]="expandable_segments:True"

    stage("2/5 · CLONAR CHARACTER FACTORY Y PREPARAR PYTHON 3.11")
    if not (REPO/".git").exists():
        run(["git","clone","--filter=blob:none","--no-checkout",
             "https://github.com/character-factory/character-factory.git",REPO],
            timeout=700)
    run(["git","checkout",CHECKPOINT],cwd=REPO,timeout=150)
    # Usamos un Python aislado: Colab suele venir con Python 3.13.
    if not shutil.which("uv"):
        run([sys.executable,"-m","pip","install","-q","uv"],timeout=360)
    uv=shutil.which("uv") or "/usr/local/bin/uv"
    run([uv,"python","install","3.11"],timeout=700)
    if not PY.exists():
        run([uv,"venv","--seed","--python","3.11",ENV],timeout=360)
    run([PY,"-c","import sys; assert sys.version_info[:2]==(3,11)"],timeout=50)

    stage("3/5 · INSTALAR PAQUETES ABIERTOS Y CHEQUEAR CUDA")
    # Cu124 y torch 2.5.1 funcionaron en el piloto anterior con Tesla T4.
    run([PY,"-m","pip","install","-q","torch==2.5.1",
         "--index-url","https://download.pytorch.org/whl/cu124"],timeout=1800,env=cleanenv)
    run([PY,"-m","pip","install","-q","-e",".[generation]"],
        cwd=REPO,timeout=1600,env=cleanenv)
    run([PY,"-c",
         "import torch; assert torch.cuda.is_available(); torch.zeros(1,device='cuda'); print('CUDA OK',torch.__version__,torch.cuda.get_device_name(0))"],
        cwd=REPO,timeout=180,env=cleanenv)
    run([BIN,"preflight"],cwd=REPO,timeout=240,env=cleanenv)
    # Revisión de espacio después de instalar paquetes (descarga de modelos ~36.4 GB).
    remaining=shutil.disk_usage("/content").free/2**30
    print("Disco antes de pesos GiB:",round(remaining,2))
    if remaining<43:
        raise RuntimeError("No queda suficiente espacio libre para modelos + export. Sin descarga.")

    stage("4/5 · GENERAR PERSONAJE Y CAPA DE PRENDA (EXPERIMENTO)")
    print("Descarga inicial oficial de modelos ~36.4GB. Puede tardar mucho y Colab puede desconectarse.")
    run([BIN,"make",PROMPT,"-o",OUTPUT,"--seed","17"],
        cwd=REPO,timeout=3600,env=cleanenv)
    glb=OUTPUT/"scene.glb"
    cfg=OUTPUT/"character.char.json"
    if not glb.is_file() or glb.stat().st_size<100000 or not cfg.is_file():
        raise RuntimeError("La generación no produjo scene.glb + character.char.json reales.")
    raw=glb.read_bytes()
    if raw[:4]!=b"glTF":raise RuntimeError("INVALID_GLTF_MAGIC")
    obj=json.loads(cfg.read_text(encoding="utf-8"))
    print("GLB real:",round(len(raw)/1024**2,2),"MiB; JSON válido:",bool(obj))

    stage("5/5 · ZIP DE PRUEBA Y DESCARGA")
    report={
      "schema":"highfly.characterfactory.probe/v1","originalRepo":"character-factory/character-factory",
      "upstreamCommit":CHECKPOINT,"softwareLicense":"Apache-2.0 (upstream)",
      "hardware":gpu.name,"vramGiB":round(vram,2),
      "characterFactoryRig":"MHR with 127 joints (NOT ClaudeCraft Rig_Medium)",
      "claudecraftNativeRigModified":False,
      "armorWearableGenerated":False,"clothingLayerShellOnly":True,
      "validatedOnWarrior":False,"commercialCreditAPIUsed":False,
      "glbSha256":hashlib.sha256(raw).hexdigest(),
      "createdUtc":datetime.now(timezone.utc).isoformat(),
      "elapsedMinutes":round((time.time()-t0)/60,1)
    }
    report_file=OUTPUT/"highfly-probe-report.json"
    report_file.write_text(json.dumps(report,indent=2,ensure_ascii=False),encoding="utf-8")
    with zipfile.ZipFile(ZIP,"w",zipfile.ZIP_DEFLATED) as z:
        z.write(glb,"scene.glb")
        z.write(cfg,"character.char.json")
        z.write(report_file,"highfly-probe-report.json")
    print("HIGHFLY ✓ PRUEBA CHARACTER FACTORY COMPLETADA:",ZIP,flush=True)
    from google.colab import files
    files.download(str(ZIP))
except Exception as exc:
    print("\n"+"!"*65)
    print("HIGHFLY FALLÓ EN:",stage_name)
    print("ERROR:",type(exc).__name__,str(exc))
    print("Si hay error, mandá las últimas líneas. No se generó una armadura compatible.")
    print("!"*65)
    raise
